In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ToxDL 2.0)

This notebook processes and standardizes the **ToxDL 2.0** dataset, a collection of peptide sequences annotated for toxicity. The workflow focuses on robust sequence parsing, label extraction from non-standard FASTA identifiers, duplicate resolution, and metadata generation to ensure consistency and traceability across downstream machine learning tasks.

- **Toxic effect / endpoint:** toxic
- **Source:** ToxDL 2.0
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Parses peptide sequences from FASTA-like files**:
  - handles non-standard characters and formatting.
- **Extracts toxicity labels from sequence identifiers**:
  - labels are parsed directly from FASTA headers using a tab-separated encoding.
- **Standardizes sequence–label pairs** into a unified binary toxicity dataset.
- **Performs duplicate sequence checks**:
  - consolidates identical sequences with consistent labels,
  - flags sequences with conflicting annotations as erroneous.
- **Generates dataset-level metadata** using a centralized raw data description file.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`
  - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "ToxDL 2.0"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
dfs = []
for file in (Path(PATH_INPUT) / name_source).glob("*"):
    df = read_fasta_with_strange_character(file)
    df["label"] = df["id"].str.split("\t").str[1].astype(int)
    df = df[["sequence", "label"]]
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
df.shape

(11677, 2)

- Checking duplicates

In [4]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [5]:
df_full.shape

(11434, 2)

In [6]:
df_errors.shape

(2, 1)

- Working with metada

In [7]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [8]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 4, 2, 0, 0),
 'download date': Timestamp('2024-08-01 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from uniprot',
 'repository or server': 'http://www.csbio.sjtu.edu.cn/bioinf/ToxDL2/',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S2001037025001230',
 'number_of_raw_sequences': 11677,
 'number_of_sequences_retained': 11434,
 'number_of_positive_sequences': 4593,
 'number_of_negative_sequences': 6841,
 'number_of_erroneous_sequences': 2,
 'modified_sequences_included': False}

- Exporting data

In [9]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [10]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)